# BestBuy.ca Review Scraper + Sentiment Analysis

**What this notebook does**
1. Opens a BestBuy.ca product page with Selenium (a real Chrome browser).
2. Cycles through the review filters (*Most Helpful, Newest, Highest Rating, Lowest Rating, Most Relevant*), clicking **Show more** until no new reviews load.
3. Extracts: Primary Key, Title, Review Text, Date (YYYY-MM-DD), Rating (/5), Source, Reviewer Name.
4. Runs sentiment analysis (overall + per-aspect, e.g. `['Design (Pos)', 'Value for Money (Neg)']`).
5. Saves a CSV + formatted Excel file, charts, and the numbers used in the business report.

**How to run:** edit the config cell (product URL), then *Kernel > Restart & Run All*.
Only publicly visible review data is collected; no login, no private data, and the scraper paces itself (random delays, one browser, no parallel requests).

> Offline test: set `USE_SAMPLE_DATA = True` in the config cell to run the analysis on synthetic demo data.

## Setup (run once in a terminal or uncomment)

In [ ]:
# pip install -r requirements.txt     # selenium, beautifulsoup4, lxml, pandas, openpyxl, vaderSentiment, matplotlib, python-docx
# You also need Google Chrome installed. Selenium downloads the matching driver automatically.

In [ ]:
# ============================================================
# STEP 1 - IMPORTS + CONFIG
# This is the only cell you normally need to edit.
# ============================================================
import re, time, random, hashlib, logging, functools, ast
from datetime import datetime, timedelta
from pathlib import Path
from collections import Counter

import pandas as pd
from bs4 import BeautifulSoup, NavigableString

# --- Which product to scrape (must have 50+ reviews) -------------------------
# Open bestbuy.ca, pick any product with 50+ reviews and paste its URL here.
# (This one had ~400 reviews when I looked it up; swap it if it is no longer listed.)
PRODUCT_URL = "https://www.bestbuy.ca/en-ca/product/google-nest-hub-2nd-gen-smart-display-with-google-assistant-chalk/15303234"

# --- Review "filters" (sort orders) the scraper will cycle through ------------
# Each filter reveals a different slice of the reviews, so together they collect more.
SORT_FILTERS = ["Most Helpful", "Newest", "Highest Rating", "Lowest Rating", "Most Relevant"]

# Words the page might use for each filter. The scraper matches on ANY of these (case-insensitive),
# so it still works if BestBuy words a label slightly differently.
SORT_SYNONYMS = {
    "Most Helpful":   ["most helpful", "helpful"],
    "Newest":         ["newest", "most recent", "recent"],
    "Highest Rating": ["highest rating", "highest rated", "highest"],
    "Lowest Rating":  ["lowest rating", "lowest rated", "lowest"],
    "Most Relevant":  ["most relevant", "relevant"],
}

# --- Behaviour ---------------------------------------------------------------
HEADLESS = True               # False = you can watch the browser (best for first run / debugging)
MAX_SHOW_MORE_CLICKS = 60     # safety cap on "Show more" clicks per filter
MIN_DELAY, MAX_DELAY = 1.5, 3.5   # random pause between actions (polite scraping)
PAGE_LOAD_TIMEOUT = 40        # seconds before a page load is considered failed
MAX_RETRIES = 3               # retries (with growing waits) for flaky steps
DEBUG_SAVE_HTML = False       # True = always save the page HTML in debug/ (useful if parsing finds 0 reviews)

# --- Offline switch ----------------------------------------------------------
# True = skip the website and run the analysis on sample_data/demo_reviews_raw.csv (SYNTHETIC data
# that exists only to prove the analysis pipeline works). Real submission must use False.
USE_SAMPLE_DATA = False

# Demo runs write into data/demo/ so they can never be mistaken for (or overwrite) real results.
OUTPUT_DIR = Path("data/demo") if USE_SAMPLE_DATA else Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR = Path("debug"); DEBUG_DIR.mkdir(exist_ok=True)

In [ ]:
# ============================================================
# STEP 2 - SMALL UTILITIES (logging, retry, polite pauses)
# ============================================================
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("bestbuy")

class BlockedError(Exception):
    """Raised when the site shows a CAPTCHA / 'access denied' page. We stop instead of hammering it."""

def clean_text(s):
    # Collapse all whitespace/newlines into single spaces.
    return re.sub(r"\s+", " ", s or "").strip()

def polite_sleep(low=None, high=None):
    # Random pause so we do not look like a bot firing requests at fixed intervals.
    time.sleep(random.uniform(low if low is not None else MIN_DELAY, high if high is not None else MAX_DELAY))

def retry(times=MAX_RETRIES, base=2.0):
    # Decorator: retry a flaky function with exponential backoff (2s, 4s, 8s ... plus jitter).
    # A BlockedError is NOT retried - retrying a block only makes it worse.
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except BlockedError:
                    raise
                except Exception as exc:
                    last_error = exc
                    wait = (base ** attempt) + random.uniform(0, 1)
                    log.warning("%s failed (attempt %d/%d): %s -> retrying in %.1fs",
                                fn.__name__, attempt, times, exc, wait)
                    time.sleep(wait)
            raise last_error
        return wrapper
    return decorator

## Parsing: how reviews are found in the page
BestBuy.ca builds its pages with JavaScript and auto-generated CSS class names that change between releases, so class-based selectors break easily. The parser instead anchors on the **date** shown on each review and works outward, which is far more stable. Rating, title, text and name are then read from the content of the card.

In [ ]:
# ============================================================
# STEP 3 - REVIEW PARSER (turns page HTML into review records)
# BestBuy.ca is a JavaScript app whose CSS class names are auto-generated and can change.
# So instead of relying on class names, we anchor on something every review has: a DATE.
#   1. find every element that holds a date
#   2. the review "card" = the biggest parent that contains exactly ONE date
#   3. inside the card, pull out rating / title / text / name using content-based rules
# ============================================================
MONTHS = (r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|"
          r"Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)")
DATE_RE = re.compile(
    r"\b\d{4}-\d{2}-\d{2}\b"                                          # 2024-01-18
    r"|\b" + MONTHS + r"\.?\s+\d{1,2},?\s+\d{4}\b"                    # Jan 18, 2024
    r"|\b\d{1,2}\s+" + MONTHS + r"\.?,?\s+\d{4}\b"                    # 18 January 2024
    r"|\b(?:a|an|\d+)\s+(?:second|minute|hour|day|week|month|year)s?\s+ago\b",  # 3 months ago
    re.I)
RATING_RE = re.compile(r"(\d(?:\.\d)?)\s*(?:out of|/|of)\s*5\b", re.I)   # "4 out of 5", "4/5"
SKIP_TAGS = {"script", "style", "noscript", "svg", "path"}
BOILERPLATE = {"helpful", "report", "read more", "show more", "verified purchase", "yes", "no",
               "was this review helpful?", "reply", "share"}

def _find_date_elements(soup):
    # Innermost elements whose (short) text is a date.
    found = []
    for el in soup.find_all(True):
        if el.name in SKIP_TAGS:
            continue
        txt = el.get_text(" ", strip=True)
        if not txt or len(txt) > 45 or not DATE_RE.search(txt):
            continue
        # skip if a child already holds the date -> we want the innermost one
        if any(DATE_RE.search(c.get_text(" ", strip=True)) for c in el.find_all(True, recursive=False)):
            continue
        found.append(el)
    return found

def _find_cards(soup):
    # For each date element, climb up while the parent still contains exactly ONE date.
    date_els = _find_date_elements(soup)
    dates_inside = Counter()
    for d in date_els:
        for ancestor in d.parents:
            dates_inside[id(ancestor)] += 1
    cards = []
    for d in date_els:
        card = d
        for ancestor in d.parents:
            if ancestor.name in ("body", "html", "[document]") or dates_inside[id(ancestor)] != 1:
                break
            card = ancestor
        cards.append((card, d))
    return cards

def _own_text(el):
    # Text directly inside this element (not inside its children).
    parts = [str(s) for s in el.find_all(string=True, recursive=False) if type(s) is NavigableString]
    return clean_text(" ".join(parts))

def _extract_rating(card):
    # 1) star widgets usually expose "4 out of 5" in aria-label / title / alt / content attributes
    for el in [card] + card.find_all(True):
        for attr in ("aria-label", "title", "alt", "content", "data-rating", "data-value"):
            v = el.get(attr)
            if v:
                m = RATING_RE.search(str(v))
                if m:
                    return float(m.group(1))
        if el.get("itemprop") == "ratingValue":
            try:
                return float(el.get("content") or el.get_text(strip=True))
            except ValueError:
                pass
    # 2) visible text such as "4/5"
    m = RATING_RE.search(card.get_text(" ", strip=True))
    if m:
        return float(m.group(1))
    # 3) last resort: count "filled" star elements
    filled = 0
    for el in card.find_all(True):
        cls = " ".join(el.get("class", [])).lower()
        if "star" in cls and any(k in cls for k in ("full", "filled", "active", "selected", "checked")):
            filled += 1
    return float(filled) if 1 <= filled <= 5 else None

def _extract_title(card, exclude):
    for el in card.find_all(["h1", "h2", "h3", "h4", "h5", "h6"]):
        t = clean_text(el.get_text(" "))
        if t and t not in exclude and not DATE_RE.fullmatch(t):
            return t[:200]
    for el in card.find_all(True):
        cls = " ".join(el.get("class", [])).lower() + " " + (el.get("data-automation") or "").lower()
        if any(k in cls for k in ("title", "headline", "heading")):
            t = clean_text(el.get_text(" "))
            if t and t not in exclude and len(t) <= 200:
                return t
    for el in card.find_all(["strong", "b"]):
        t = clean_text(el.get_text(" "))
        if t and t not in exclude and len(t) <= 120:
            return t
    return ""

def _extract_review_text(card, exclude):
    # The review body = the element with the most text of its own (excluding date/title/boilerplate).
    best_el, best_txt = None, ""
    for el in card.find_all(True):
        if el.name in SKIP_TAGS:
            continue
        t = _own_text(el)
        if not t or t in exclude or t.lower() in BOILERPLATE:
            continue
        if len(t) <= 45 and DATE_RE.search(t):
            continue
        if len(t) > len(best_txt):
            best_el, best_txt = el, t
    # If the review is split over several <p> tags, join them.
    if best_el is not None and best_el.name == "p" and best_el.parent is not None:
        paras = [_own_text(p) for p in best_el.parent.find_all("p", recursive=False)]
        paras = [p for p in paras if p and p not in exclude]
        if len(paras) >= 2:
            best_txt = " ".join(paras)
    return best_txt

def _extract_name(card, exclude, date_el=None):
    el = card.find(attrs={"itemprop": "author"})
    if el:
        t = clean_text(el.get_text(" "))
        if t:
            return t[:60]
    for el in card.find_all(True):
        cls = " ".join(el.get("class", [])).lower() + " " + (el.get("data-automation") or "").lower()
        if any(k in cls for k in ("author", "reviewer", "nickname", "username", "user-name", "display-name")):
            t = clean_text(el.get_text(" "))
            if t and len(t) <= 60 and t not in exclude and not DATE_RE.search(t):
                return t
    # Fallback: a short word-only text right next to the date is usually the reviewer's name.
    if date_el is not None and date_el.parent is not None:
        for sib in date_el.parent.find_all(True):
            t = _own_text(sib)
            if (sib is not date_el and t and t not in exclude and 2 <= len(t) <= 40
                    and not DATE_RE.search(t) and not re.search(r"\d", t) and t.lower() not in BOILERPLATE):
                return t
    return "Anonymous"

def _fallback_title(card, exclude, body_text):
    # Fallback: the first short text that appears BEFORE the review body is usually the title.
    for el in card.find_all(True):
        if el.name in SKIP_TAGS:
            continue
        t = _own_text(el)
        if not t:
            continue
        if t == body_text:
            break
        if (t not in exclude and 2 <= len(t) <= 100 and not DATE_RE.search(t)
                and t.lower() not in BOILERPLATE):
            return t
    return ""

def _extract_source(card):
    # Some reviews are syndicated ("Originally posted on <Brand>"). Default = BestBuy.ca.
    m = re.search(r"(?i:originally posted on|review from|source:)\s+([A-Z][A-Za-z0-9.&'-]*(?: [A-Z][A-Za-z0-9.&'-]*){0,3})",
                  card.get_text(" ", strip=True))
    return m.group(1) if m else "BestBuy.ca"

def _extract_dom_id(card):
    # Use the site's own review id when the card carries one.
    for attr in ("data-review-id", "data-reviewid", "data-id", "id"):
        v = card.get(attr)
        if v and re.fullmatch(r"[A-Za-z0-9_-]{6,40}", str(v)):
            return str(v)
    return None

def content_key(name, date_raw, title, text):
    # Stable fingerprint of a review; used to remove duplicates across filters/pages.
    blob = "|".join(clean_text(x).lower() for x in (name, date_raw, title, text[:120]))
    return "BB-" + hashlib.sha1(blob.encode("utf-8")).hexdigest()[:12].upper()

def parse_reviews(html):
    # Main parser: HTML string -> list of review dicts.
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style", "noscript"]):
        t.decompose()
    reviews, seen = [], set()
    for card, date_el in _find_cards(soup):
        card_text = card.get_text(" ", strip=True)
        if len(card_text) < 30 or len(card_text) > 6000:      # too small / the whole page -> not a review card
            continue
        date_raw = clean_text(date_el.get_text(" "))
        rating = _extract_rating(card)
        title = _extract_title(card, {date_raw})
        text = _extract_review_text(card, {date_raw, title})
        if not text:
            continue
        name = _extract_name(card, {date_raw, title, text}, date_el)
        if not title:
            title = _fallback_title(card, {date_raw, name}, text)
        key = content_key(name, date_raw, title, text)
        if key in seen:
            continue
        seen.add(key)
        reviews.append({"key": key, "dom_id": _extract_dom_id(card), "title": title, "review_text": text,
                        "date_raw": date_raw, "rating": rating, "source": _extract_source(card),
                        "reviewer_name": name})
    return reviews

## Browser automation (filters, "Show more" paging, retries, block detection)

In [ ]:
# ============================================================
# STEP 4 - BROWSER AUTOMATION (Selenium)
# Everything that touches the live website lives here. Every step is wrapped so a failure is
# logged and the run continues with whatever it already collected.
# ============================================================
SHOW_MORE_PHRASES = ["show more", "load more", "more reviews", "view more", "see more reviews", "show all reviews"]
OPEN_REVIEWS_PHRASES = ["see all reviews", "view all reviews", "all reviews", "customer reviews", "read reviews", "reviews"]
BLOCK_MARKERS = ["captcha", "access denied", "unusual traffic", "verify you are a human",
                 "are you a robot", "request blocked", "pardon our interruption"]
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
              "Chrome/124.0.0.0 Safari/537.36")

# One JavaScript call that finds visible clickable things whose text contains any phrase.
# Doing this in JS (instead of looping in Python) is much faster and does not depend on CSS classes.
FIND_CLICKABLES_JS = r"""
const phrases = arguments[0], maxLen = arguments[1];
const sel = "button, a, [role='button'], [role='option'], [role='menuitem'], [role='tab'], [role='combobox'], li, label";
const out = [];
for (const el of document.querySelectorAll(sel)) {
  const t = (el.innerText || "").replace(/\s+/g, " ").trim().toLowerCase();
  if (!t || t.length > maxLen) continue;
  if (!phrases.some(p => t.includes(p))) continue;
  const r = el.getBoundingClientRect();
  if (r.width === 0 || r.height === 0) continue;
  out.push(el);
}
return out.filter(e => !out.some(o => o !== e && e.contains(o)));   // keep the innermost match only
"""

def build_driver(headless=HEADLESS):
    # Start Chrome. Selenium 4.6+ downloads the matching chromedriver automatically (needs internet + Chrome installed).
    try:
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
    except ImportError as exc:
        raise RuntimeError("Selenium is not installed. Run: pip install -r requirements.txt") from exc
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    for arg in ("--window-size=1400,1000", "--lang=en-CA", "--no-sandbox", "--disable-dev-shm-usage",
                "--disable-blink-features=AutomationControlled", f"--user-agent={USER_AGENT}"):
        opts.add_argument(arg)
    opts.page_load_strategy = "eager"          # do not wait for every image/tracker to finish
    try:
        driver = webdriver.Chrome(options=opts)
    except Exception as exc:
        raise RuntimeError("Could not start Chrome. Make sure Google Chrome is installed and you are online "
                           f"(Selenium fetches the driver itself). Original error: {exc}") from exc
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    return driver

def check_blocked(driver):
    # Look only at the page title + the first visible text (not scripts, which often mention 'captcha').
    try:
        body = driver.execute_script("return (document.body && document.body.innerText || '').slice(0, 1500);")
        sample = ((driver.title or "") + " " + body).lower()
    except Exception:
        return
    hit = next((m for m in BLOCK_MARKERS if m in sample), None)
    if hit:
        raise BlockedError(f"Block page detected (matched '{hit}')")

@retry()
def open_page(driver, url):
    # Load the page; retried with backoff if the network/page hiccups.
    from selenium.webdriver.support.ui import WebDriverWait
    driver.get(url)
    WebDriverWait(driver, 20).until(lambda d: d.execute_script("return document.readyState") in ("interactive", "complete"))
    polite_sleep(2, 4)
    check_blocked(driver)

def find_clickables(driver, phrases, max_len=40):
    return driver.execute_script(FIND_CLICKABLES_JS, [p.lower() for p in phrases], max_len) or []

def safe_click(driver, el):
    # Scroll the element into view and click it; fall back to a JavaScript click if something overlaps it.
    try:
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
        time.sleep(0.4)
        el.click()
        return True
    except Exception:
        try:
            driver.execute_script("arguments[0].click();", el)
            return True
        except Exception as exc:
            log.debug("click failed: %s", exc)
            return False

def element_key(el):
    try:
        return (el.get_attribute("outerHTML") or "")[:300]
    except Exception:
        return None

def count_reviews(driver):
    return len(parse_reviews(driver.page_source))

def dismiss_overlays(driver):
    # Close cookie / promo pop-ups if present. Nothing happens if there are none.
    for phrases in (["accept all", "accept cookies"], ["no thanks", "not now", "got it"]):
        try:
            els = find_clickables(driver, phrases, max_len=25)
            if els:
                safe_click(driver, els[0])
                time.sleep(0.8)
        except Exception as exc:
            log.debug("overlay step skipped: %s", exc)

def scroll_through_page(driver, steps=10, pause=0.6):
    # Review sections are lazy-loaded: scrolling down in steps makes them appear.
    for i in range(1, steps + 1):
        driver.execute_script("window.scrollTo(0, (document.body.scrollHeight / arguments[0]) * arguments[1]);", steps, i)
        time.sleep(pause)

def wait_for_growth(driver, before, timeout=8):
    # After clicking "Show more", wait until the number of parsed reviews goes up.
    end = time.time() + timeout
    while time.time() < end:
        time.sleep(0.8)
        now = count_reviews(driver)
        if now > before:
            return now
    return before

def prepare_page(driver):
    # Get the page into a state where reviews are visible.
    dismiss_overlays(driver)
    scroll_through_page(driver)
    if count_reviews(driver) == 0:
        # Reviews may sit behind a "See all reviews" / "Reviews" tab. Try each candidate until reviews appear.
        for el in find_clickables(driver, OPEN_REVIEWS_PHRASES, max_len=40)[:6]:
            if safe_click(driver, el):
                time.sleep(2)
                scroll_through_page(driver, steps=4, pause=0.5)
                if count_reviews(driver) > 0:
                    break

def apply_sort(driver, label):
    # Apply one of the review filters. Returns True if a control was found and used.
    synonyms = SORT_SYNONYMS.get(label, [label.lower()])
    # (a) a normal <select> dropdown
    try:
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import Select
        for sel in driver.find_elements(By.TAG_NAME, "select"):
            select = Select(sel)
            for opt in select.options:
                t = clean_text(opt.text).lower()
                if t and any(s in t for s in synonyms):
                    select.select_by_visible_text(opt.text)
                    polite_sleep(1.5, 2.5)
                    return True
    except Exception as exc:
        log.debug("select-style sort failed: %s", exc)
    # (b) a custom dropdown: click "Sort by", then click the option
    try:
        openers = find_clickables(driver, ["sort by", "sort"], max_len=40)
        if openers and safe_click(driver, openers[0]):
            time.sleep(0.8)
            options = find_clickables(driver, synonyms, max_len=40)
            if options and safe_click(driver, options[0]):
                polite_sleep(1.5, 2.5)
                return True
    except Exception as exc:
        log.debug("custom sort failed: %s", exc)
    return False

def load_more_reviews(driver, max_clicks=MAX_SHOW_MORE_CLICKS):
    # Click "Show more" until no new reviews appear. We judge a button by its EFFECT: if clicking it
    # does not increase the review count, it is probably a different "show more" (e.g. specs) and is skipped.
    clicks, consecutive_fail, fails = 0, 0, Counter()
    current = count_reviews(driver)
    log.info("  reviews visible before paging: %d", current)
    while clicks < max_clicks and consecutive_fail < 6:
        candidates = [(c, element_key(c)) for c in find_clickables(driver, SHOW_MORE_PHRASES, max_len=40)]
        candidates = [(c, k) for c, k in candidates if k and fails[k] < 2]
        if not candidates:
            break
        el, key = candidates[0]
        if not safe_click(driver, el):
            fails[key] += 1; consecutive_fail += 1
            continue
        clicks += 1
        new = wait_for_growth(driver, current)
        if new > current:
            log.info("  click %d -> %d reviews", clicks, new)
            current, consecutive_fail = new, 0
            polite_sleep()
        else:
            fails[key] += 1; consecutive_fail += 1
    return current

def expand_read_more(driver):
    # Some long reviews are truncated with "Read more". Expand them, but never leave the product page.
    url_before = driver.current_url
    try:
        for _ in range(3):
            els = find_clickables(driver, ["read more", "show full review", "see full review"], max_len=25)
            if not els:
                break
            for el in els[:300]:
                safe_click(driver, el)
            if driver.current_url != url_before:
                driver.back(); time.sleep(2)
                break
    except Exception as exc:
        log.debug("read-more step skipped: %s", exc)

In [ ]:
# ============================================================
# STEP 5 - SCRAPE ONE FILTER, THEN ALL FILTERS
# For every filter: reload page -> apply filter -> click "Show more" until done -> parse.
# Reviews from all filters are merged and de-duplicated.
# ============================================================
def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

def scrape_one_filter(driver, url, label):
    log.info("=== Filter: %s ===", label)
    open_page(driver, url)                      # fresh page each time = clean state
    prepare_page(driver)
    if apply_sort(driver, label):
        log.info("  filter applied")
    else:
        log.warning("  could not find the '%s' control - using the page's default order", label)
    load_more_reviews(driver)
    expand_read_more(driver)
    check_blocked(driver)
    html = driver.page_source
    rows = parse_reviews(html)
    if DEBUG_SAVE_HTML or not rows:
        path = DEBUG_DIR / f"page_{slug(label)}.html"
        path.write_text(html, encoding="utf-8")
        log.info("  saved page HTML to %s", path)
    log.info("  parsed %d reviews", len(rows))
    return rows

def scrape_reviews(url, filters):
    # Loop over all filters, merge + de-duplicate, and save a partial CSV after each filter
    # so a crash later never loses what was already collected.
    merged = {}
    driver = None
    try:
        driver = build_driver()
        for label in filters:
            try:
                rows = scrape_one_filter(driver, url, label)
            except BlockedError as exc:
                log.error("BLOCKED: %s. Stopping early and keeping the %d reviews collected so far.", exc, len(merged))
                break
            except Exception as exc:
                log.error("Filter '%s' failed (%s). Moving on to the next filter.", label, exc)
                continue
            new = 0
            for r in rows:
                if r["key"] in merged:
                    merged[r["key"]]["filters_seen_in"].append(label)
                else:
                    merged[r["key"]] = {**r, "filters_seen_in": [label]}
                    new += 1
            log.info("  +%d new reviews (total unique: %d)", new, len(merged))
            if merged:
                pd.DataFrame(list(merged.values())).to_csv(OUTPUT_DIR / "raw_reviews_partial.csv", index=False)
            polite_sleep(3, 6)
    finally:
        if driver is not None:
            try:
                driver.quit()
            except Exception:
                pass
    df = pd.DataFrame(list(merged.values()))
    if not df.empty:
        df["filters_seen_in"] = df["filters_seen_in"].map(lambda x: ", ".join(dict.fromkeys(x)))
    return df

## Run it

In [ ]:
# ============================================================
# STEP 6 - RUN THE SCRAPE (or load the offline demo data)
# ============================================================
if USE_SAMPLE_DATA:
    log.warning("USE_SAMPLE_DATA=True -> using SYNTHETIC demo data, not real BestBuy reviews.")
    raw_df = pd.read_csv("sample_data/demo_reviews_raw.csv")
else:
    raw_df = scrape_reviews(PRODUCT_URL, SORT_FILTERS)

if raw_df.empty:
    raise RuntimeError(
        "0 reviews were collected. What to do:\n"
        "  1. Set HEADLESS = False and re-run to watch the browser - is the product page / reviews section visible?\n"
        "  2. Open debug/page_*.html in a browser or editor and check whether reviews are in the HTML.\n"
        "  3. If a CAPTCHA / 'access denied' page appeared, see the anti-blocking section of the README.\n"
        "  4. Send me the debug HTML and I will adjust the parser.")

log.info("Collected %d unique reviews", len(raw_df))
raw_df.head()

## Clean the data

In [ ]:
# ============================================================
# STEP 7 - CLEAN + STANDARDISE into the required fields
# Primary Key | Title | Review Text | Date (YYYY-MM-DD) | Rating (1-5) | Source | Reviewer Name
# ============================================================
def normalize_date(raw):
    # Accepts "2024-01-18", "Jan 18, 2024", "18 January 2024" or "3 months ago" -> "YYYY-MM-DD".
    # (Relative dates are converted using today's date, so they are approximate.)
    raw = clean_text(str(raw)) if pd.notna(raw) else ""
    m = DATE_RE.search(raw)
    if not m:
        return ""
    s = m.group(0)
    rel = re.match(r"(a|an|\d+)\s+(second|minute|hour|day|week|month|year)s?\s+ago", s, re.I)
    if rel:
        n = 1 if rel.group(1).lower() in ("a", "an") else int(rel.group(1))
        days = {"second": 0, "minute": 0, "hour": 0, "day": 1, "week": 7, "month": 30, "year": 365}[rel.group(2).lower()]
        return (datetime.now() - timedelta(days=n * days)).strftime("%Y-%m-%d")
    dt = pd.to_datetime(s, errors="coerce")
    return dt.strftime("%Y-%m-%d") if pd.notna(dt) else ""

df = pd.DataFrame({
    "Primary Key":   raw_df["dom_id"].where(raw_df["dom_id"].notna(), raw_df["key"]),
    "Title":         raw_df["title"].fillna("").map(clean_text),
    "Review Text":   raw_df["review_text"].fillna("").map(clean_text),
    "Date":          raw_df["date_raw"].map(normalize_date),
    "Rating":        pd.to_numeric(raw_df["rating"], errors="coerce"),
    "Source":        raw_df["source"].fillna("BestBuy.ca"),
    "Reviewer Name": raw_df["reviewer_name"].fillna("Anonymous"),
})

# Rating must be 1-5; anything else becomes blank instead of polluting the analysis.
df.loc[~df["Rating"].between(1, 5), "Rating"] = pd.NA
# Primary keys must be unique: if a site id repeats, fall back to our content fingerprint.
dup = df["Primary Key"].duplicated(keep=False)
df.loc[dup, "Primary Key"] = raw_df.loc[dup, "key"]
df = df.drop_duplicates("Primary Key")
df = df[df["Review Text"].str.len() > 0].sort_values("Date", ascending=False).reset_index(drop=True)

log.info("Clean dataset: %d reviews | missing rating: %d | missing date: %d",
         len(df), df["Rating"].isna().sum(), (df["Date"] == "").sum())
if len(df) < 50:
    log.warning("Only %d reviews - the brief asks for a product with at least 50. Try a product with more reviews "
                "or check that 'Show more' paging worked (see the log above).", len(df))
df.head()

## Sentiment analysis

In [ ]:
# ============================================================
# STEP 8 - SENTIMENT ANALYSIS
# Two layers, so the output looks like the example in the brief:
#   (1) overall polarity of each review  -> Positive / Neutral / Negative   (VADER)
#   (2) aspect-level sentiment           -> e.g. ['Design (Pos)', 'Ease of Use (Pos)', 'Value for Money (Neg)']
# VADER is a rule-based model built for short, informal text like reviews - no training or GPU needed.
# ============================================================
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _vader = SentimentIntensityAnalyzer()
    ENGINE = "VADER"
    def polarity(text):
        return _vader.polarity_scores(text)["compound"]          # -1 (very negative) ... +1 (very positive)
except ImportError:
    # Fallback so the notebook never crashes if vaderSentiment is missing (install it for better accuracy).
    ENGINE = "built-in word-list fallback (pip install vaderSentiment for better accuracy)"
    _POS = set("good great excellent love loved amazing awesome perfect nice easy happy pretty fantastic best "
               "works worked reliable sturdy recommend recommended satisfied beautiful fast useful solid worth fine "
               "sleek bright clear responsive".split())
    _NEG = set("bad poor terrible awful hate hated broken broke worst slow useless disappointed disappointing cheap "
               "flimsy defective difficult confusing waste problem problems issue issues stopped fail failed faulty "
               "annoying expensive overpriced laggy frustrating late damaged tinny weak freezes".split())
    _NEGATORS = {"not", "no", "never", "dont", "don't", "isn't", "wasn't", "doesn't", "didn't", "cant", "can't", "won't"}
    def polarity(text):
        tokens = re.findall(r"[a-z']+", text.lower())
        if not tokens:
            return 0.0
        score = 0
        for i, tok in enumerate(tokens):
            val = 1 if tok in _POS else -1 if tok in _NEG else 0
            if val and any(t in _NEGATORS for t in tokens[max(0, i - 3):i]):
                val = -val                                          # "not good" flips the sign
            score += val
        return max(-1.0, min(1.0, score / (len(tokens) ** 0.5)))
log.info("Sentiment engine: %s", ENGINE)

POS_T, NEG_T = 0.05, -0.05        # standard VADER cut-offs
def label_from_score(score):
    return "Positive" if score >= POS_T else "Negative" if score <= NEG_T else "Neutral"
def short_tag(label):
    return {"Positive": "Pos", "Negative": "Neg", "Neutral": "Neu"}[label]

# Keywords that map a sentence to a business aspect (edit/extend for other product types).
ASPECT_KEYWORDS = {
    "Design":                ["design", "looks", "look", "sleek", "style", "stylish", "colour", "color", "size", "compact",
                              "bulky", "slim", "pretty", "beautiful", "build"],
    "Quality & Durability":  ["quality", "sturdy", "solid", "flimsy", "durable", "broke", "broken", "defective",
                              "stopped working", "died", "fell apart", "cheaply made", "feels cheap"],
    "Ease of Use":           ["easy", "easily", "simple", "setup", "set up", "install", "intuitive", "user friendly",
                              "confusing", "complicated", "difficult", "instructions", "app"],
    "Value for Money":       ["price", "value", "worth", "money", "affordable", "expensive", "deal", "overpriced",
                              "pricey", "cost", "sale"],
    "Performance":           ["fast", "slow", "lag", "laggy", "speed", "performance", "works", "working", "powerful",
                              "responsive", "glitch", "freeze", "freezes", "crash", "reliable", "buggy"],
    "Display & Sound":       ["screen", "display", "sound", "speaker", "audio", "volume", "bass", "picture",
                              "brightness", "resolution"],
    "Connectivity":          ["wifi", "wi-fi", "bluetooth", "connect", "connection", "pairing", "pair", "sync",
                              "signal", "disconnect", "disconnecting", "disconnects"],
    "Battery & Power":       ["battery", "charge", "charging", "power", "plug", "cord"],
    "Customer Service & Delivery": ["service", "support", "delivery", "shipping", "shipped", "arrived", "return",
                              "returning", "refund", "warranty", "staff"],
    "Features":              ["feature", "features", "assistant", "voice", "smart", "camera"],
    "Overall Satisfaction":  ["love", "great", "happy", "satisfied", "disappointed", "recommend", "perfect",
                              "excellent", "terrible", "awful", "stars", "frustrating"],
}
ASPECT_PATTERNS = {a: re.compile(r"\b(?:" + "|".join(re.escape(k) for k in kws) + r")\b", re.I)
                   for a, kws in ASPECT_KEYWORDS.items()}

def split_sentences(text):
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+|\n+", text) if p.strip()]

def analyse_review(title, text):
    # Returns (overall_score, overall_label, [(aspect, label), ...]) for one review.
    full = f"{title}. {text}" if title else text
    score = polarity(full)
    overall = label_from_score(score)
    sentences = split_sentences(text)
    if title:
        sentences.insert(0, title)
    aspects = []
    for aspect, pattern in ASPECT_PATTERNS.items():
        hits = [s for s in sentences if pattern.search(s)]
        if hits:
            a_score = sum(polarity(s) for s in hits) / len(hits)
            aspects.append((aspect, label_from_score(a_score)))
    if not aspects:
        aspects = [("General Satisfaction", overall)]
    return score, overall, aspects

results = [analyse_review(t, x) for t, x in zip(df["Title"], df["Review Text"])]
df["Sentiment_Score"] = [round(r[0], 3) for r in results]
df["Sentiment_Label"] = [r[1] for r in results]
aspect_pairs = [r[2] for r in results]                                   # kept as python lists for the summaries
df["Aspect_Sentiment"] = [str([f"{a} ({short_tag(l)})" for a, l in pairs]) for pairs in aspect_pairs]

# Sanity check: flag reviews where the star rating and the text disagree (sarcasm, wrong star click, etc.)
def mismatch(rating, label):
    if pd.isna(rating):
        return ""
    if rating >= 4 and label == "Negative":
        return "High rating / negative text"
    if rating <= 2 and label == "Positive":
        return "Low rating / positive text"
    return ""
df["Rating_vs_Text_Flag"] = [mismatch(r, l) for r, l in zip(df["Rating"], df["Sentiment_Label"])]

print(df["Sentiment_Label"].value_counts().to_string())
df[["Title", "Review Text", "Sentiment_Label", "Aspect_Sentiment"]].head(5)

## Insights and charts

In [ ]:
# ============================================================
# STEP 9 - SUMMARY TABLES, CHARTS AND BUSINESS INSIGHTS
# ============================================================
import matplotlib.pyplot as plt

GENERIC_ASPECTS = {"Overall Satisfaction", "General Satisfaction"}   # too vague to be "drivers"

def build_aspect_table(pairs_list):
    # Count how often each aspect is mentioned positively / negatively / neutrally.
    counts = Counter()
    for pairs in pairs_list:
        for aspect, label in pairs:
            counts[(aspect, label)] += 1
    rows = []
    for aspect in sorted({a for a, _ in counts}):
        pos, neg, neu = counts[(aspect, "Positive")], counts[(aspect, "Negative")], counts[(aspect, "Neutral")]
        total = pos + neg + neu
        rows.append({"Aspect": aspect, "Positive": pos, "Negative": neg, "Neutral": neu,
                     "Mentions": total, "Net Score": round((pos - neg) / total, 2)})
    return pd.DataFrame(rows).sort_values("Mentions", ascending=False).reset_index(drop=True)

aspect_df = build_aspect_table(aspect_pairs)
drivers = aspect_df[~aspect_df["Aspect"].isin(GENERIC_ASPECTS)]
top_satisfaction = drivers[drivers["Positive"] > 0].sort_values(["Positive", "Net Score"], ascending=False).head(3)
top_dissatisfaction = drivers[drivers["Negative"] > 0].sort_values(["Negative", "Net Score"], ascending=[False, True]).head(3)

sent_counts = df["Sentiment_Label"].value_counts().reindex(["Positive", "Neutral", "Negative"], fill_value=0)
summary = {
    "Reviews analysed": len(df),
    "Average rating (out of 5)": round(df["Rating"].astype(float).mean(), 2),
    "Date range": f"{df.loc[df['Date'] != '', 'Date'].min()} to {df.loc[df['Date'] != '', 'Date'].max()}",
    "% Positive": round(100 * sent_counts["Positive"] / len(df), 1),
    "% Neutral": round(100 * sent_counts["Neutral"] / len(df), 1),
    "% Negative": round(100 * sent_counts["Negative"] / len(df), 1),
    "Rating/text mismatches": int((df["Rating_vs_Text_Flag"] != "").sum()),
    "Sentiment engine": ENGINE,
}
summary_df = pd.DataFrame({"Metric": list(summary.keys()), "Value": [str(v) for v in summary.values()]})

# ---------- charts (saved as PNG so the report can reuse them) ----------
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(sent_counts.index, sent_counts.values, color=["#4caf50", "#ffc107", "#e53935"])
ax.set_title("Review sentiment"); ax.set_ylabel("Reviews")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "chart_sentiment.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(5, 3.2))
rating_counts = df["Rating"].dropna().astype(int).value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
ax.bar(rating_counts.index.astype(str), rating_counts.values, color="#1976d2")
ax.set_title("Star ratings"); ax.set_xlabel("Stars"); ax.set_ylabel("Reviews")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "chart_ratings.png", dpi=150); plt.show()

plot_df = drivers.sort_values("Mentions").tail(8)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.barh(plot_df["Aspect"], plot_df["Positive"], color="#4caf50", label="Positive")
ax.barh(plot_df["Aspect"], -plot_df["Negative"], color="#e53935", label="Negative")
ax.axvline(0, color="black", lw=0.8); ax.set_title("What reviewers talk about (positive vs negative mentions)")
ax.legend(); plt.tight_layout(); plt.savefig(OUTPUT_DIR / "chart_aspects.png", dpi=150); plt.show()

# ---------- printed insights ----------
def example_reviews(aspect, label, n=2):
    # The strongest example reviews for an aspect (most negative first for 'Negative', most positive first for 'Positive').
    idx = [i for i, pairs in enumerate(aspect_pairs) if (aspect, label) in pairs]
    sub = df.iloc[idx].sort_values("Sentiment_Score", ascending=(label == "Negative")).head(n)
    return [f'"{t[:160]}..." (rating {r})' for t, r in zip(sub["Review Text"], sub["Rating"])]

print(summary_df.to_string(index=False))
print("\nTop drivers of SATISFACTION:")
for _, r in top_satisfaction.iterrows():
    print(f"  - {r['Aspect']}: {r['Positive']} positive vs {r['Negative']} negative mentions")
print("\nTop drivers of DISSATISFACTION:")
for _, r in top_dissatisfaction.iterrows():
    print(f"  - {r['Aspect']}: {r['Negative']} negative vs {r['Positive']} positive mentions")
    for ex in example_reviews(r["Aspect"], "Negative", 1):
        print("      e.g.", ex)
aspect_df

## Export deliverables

In [ ]:
# ============================================================
# STEP 10 - EXPORT (CSV + formatted Excel)
# ============================================================
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

FINAL_COLUMNS = ["Primary Key", "Title", "Review Text", "Date", "Rating", "Source", "Reviewer Name",
                 "Sentiment_Score", "Sentiment_Label", "Aspect_Sentiment", "Rating_vs_Text_Flag"]
csv_path = OUTPUT_DIR / "bestbuy_reviews_sentiment.csv"
xlsx_path = OUTPUT_DIR / "bestbuy_reviews_sentiment.xlsx"

df[FINAL_COLUMNS].to_csv(csv_path, index=False, encoding="utf-8-sig")      # utf-8-sig so Excel shows accents correctly

# Write 3 sheets, then style them.
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df[FINAL_COLUMNS].to_excel(writer, sheet_name="Reviews", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    aspect_df.to_excel(writer, sheet_name="Aspect Summary", index=False)

wb = load_workbook(xlsx_path)
header_fill = PatternFill("solid", fgColor="C6E0B4")                          # green header like the example in the brief
fills = {"Positive": PatternFill("solid", fgColor="C6EFCE"),
         "Negative": PatternFill("solid", fgColor="FFC7CE"),
         "Neutral":  PatternFill("solid", fgColor="FFEB9C")}
widths = {"Primary Key": 20, "Title": 28, "Review Text": 70, "Date": 12, "Rating": 8, "Source": 16, "Reviewer Name": 18,
          "Sentiment_Score": 15, "Sentiment_Label": 15, "Aspect_Sentiment": 48, "Rating_vs_Text_Flag": 26}

ws = wb["Reviews"]
for col_idx, name in enumerate(FINAL_COLUMNS, start=1):
    letter = get_column_letter(col_idx)
    ws.column_dimensions[letter].width = widths[name]
    ws[f"{letter}1"].fill = header_fill
    ws[f"{letter}1"].font = Font(bold=True)
label_col = FINAL_COLUMNS.index("Sentiment_Label") + 1
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical="top")
    lab = row[label_col - 1].value
    if lab in fills:
        row[label_col - 1].fill = fills[lab]
ws.freeze_panes = "A2"
ws.auto_filter.ref = ws.dimensions

for name in ("Summary", "Aspect Summary"):
    sheet = wb[name]
    for cell in sheet[1]:
        cell.fill, cell.font = header_fill, Font(bold=True)
    for col in sheet.columns:
        sheet.column_dimensions[col[0].column_letter].width = max(14, min(45, max(len(str(c.value or "")) for c in col) + 2))
wb.save(xlsx_path)

log.info("Saved: %s and %s", csv_path, xlsx_path)
df[FINAL_COLUMNS].head()

## Solutions for scraping challenges (summary - full detail in the report and README)

| Challenge | What the code already does | What to add if you scale up |
|---|---|---|
| **Anti-bot detection / IP blocking** | Real Chrome, realistic user-agent, random 1.5-3.5 s delays, one request stream, 3-6 s rest between filters | Rotating **residential** proxies (`--proxy-server=...`), one IP per session, lower request rate |
| **CAPTCHA** | `check_blocked()` detects CAPTCHA / "access denied" pages and **stops cleanly**, keeping data already collected | Do **not** try to bypass it. Slow down, wait and retry later, use the site's official data/partner feed, or a licensed review-data provider |
| **Flaky pages / timeouts** | `@retry` with exponential backoff (2s, 4s, 8s + jitter) on page loads; each filter is isolated so one failure does not kill the run | Resume from `data/raw_reviews_partial.csv` |
| **Lazy-loaded content / "Show more"** | Scrolls the page, clicks "Show more" until the review count stops growing, expands "Read more" | - |
| **Site layout changes** | Content-based parser (anchors on dates, not CSS classes); saves page HTML to `debug/` if 0 reviews found | Re-point the parser using the saved HTML |
| **Duplicates across filters** | Content fingerprint (`BB-...`) removes repeats | - |
| **Legal / policy** | Public data only, no login, polite rate | Check bestbuy.ca Terms of Use and robots.txt before running at scale |